# PS1 — data currency and horizon arithmetic

**Run this in Databricks.** It needs Unity Catalog (`mars_dev`), so it cannot run in the
SageMaker PS1 notebooks — those use a local Spark over the S3 exports and have no catalog.

Read-only. Nothing is written anywhere.

**Run All**, then copy the block printed by the last cell and send it back.

| Cell | What it answers |
|---|---|
| 1 | Config — set the date the feed is claimed to reach |
| 2 | Is every source PS1 reads actually loaded to that date, UC and S3 separately |
| 3 | What the label source itself covers, per fleet |
| 4 | Base rate and required AUC at 1 / 3 / 7 / 30-day horizons, without training anything |
| 5 | The paste-back summary |

Cell 4 is the one that decides the horizon work. The contract pair (precision 0.90,
recall 0.80) forces a required AUC that moves with the base rate, so knowing the base rate
at each horizon tells us which horizons are worth a training run and which are arithmetically
out of reach — before spending a full ETL and train on them.


In [ ]:
# ── CELL 1 ─ config ───────────────────────────────────────────────────────────
# The only line you may want to change is EXPECTED_THROUGH.
import datetime as _dt
from statistics import NormalDist

EXPECTED_THROUGH = _dt.date(2026, 9, 8)      # the date the feed is claimed to reach
CATALOG   = "mars_dev"
GOLD_S3   = "s3://cubic-mars-pm-s3-datalake-dev-gold-170202974600/chicago/gold"
SILVER_S3 = "s3://cubic-mars-pm-s3-datalake-dev-gold-170202974600/chicago/silver"

FLEETS   = ["GATE", "TVM", "VALIDATOR"]
HORIZONS = [1, 3, 7]                         # 24h / shipped default / contracted
# 30-day dropped on instruction, 24-Sep: not required, and its cross-join is 30x the
# episode rows, which is the slowest thing in this notebook. Add 30 back here if asked;
# it was already measured as degenerate -- episodes x 30 exceeds the device-day population.
SESSION_GAP_DAYS = 3                         # matches PS1_EVENT_SESSION_GAP_DAYS default

# Contract acceptance, Annexure 1 Table 13
REQ_PRECISION = 0.90
REQ_RECALL    = 0.80

from pyspark.sql import functions as F
from pyspark.sql.window import Window

print(f"checking against {EXPECTED_THROUGH}, catalog {CATALOG}")


In [ ]:
# ── CELL 2 ─ preflight: does each table EXIST, and is the name right? ────────
# A read that raises AnalysisException says nothing about WHY. "The table was never
# built", "it is in another schema" and "I spelled it wrong" all look identical, and
# the first run of this notebook returned three of them with no way to tell which.
print("=" * 108)
print("0. PREFLIGHT — table existence, before anything is read")
print("=" * 108)

CATALOGUE = {}
for _sch in ("silver", "gold"):
    try:
        rows = spark.sql(f"SHOW TABLES IN {CATALOG}.{_sch}").collect()
        CATALOGUE[_sch] = sorted(r["tableName"] for r in rows)
        print(f"  {CATALOG}.{_sch}: {len(CATALOGUE[_sch])} tables")
    except Exception as exc:
        CATALOGUE[_sch] = []
        print(f"  {CATALOG}.{_sch}: CANNOT LIST ({type(exc).__name__}) — existence "
              f"cannot be distinguished from misnaming for this schema")


def _tokens(name):
    return set(t for t in name.lower().replace("-", "_").split("_") if len(t) > 2)


def near(name, schema, k=4):
    """Names in the schema sharing the most tokens -- a misnaming shows up here."""
    want = _tokens(name)
    scored = [(len(want & _tokens(t)), t) for t in CATALOGUE.get(schema, [])]
    return [t for n, t in sorted(scored, reverse=True)[:k] if n > 0]


MISSING, MISNAMED = [], []
print()
for layer, tbl, datecol, tier in SOURCES:
    known = CATALOGUE.get(layer)
    if not known:
        continue
    if tbl in known:
        continue
    cand = near(tbl, layer)
    if cand:
        MISNAMED.append((layer, tbl, cand))
        print(f"  NOT FOUND  {layer}.{tbl}")
        print(f"             closest names that DO exist: {', '.join(cand)}")
    else:
        MISSING.append((layer, tbl))
        print(f"  NOT FOUND  {layer}.{tbl}   (nothing similar in {layer} — genuinely absent)")

if not MISSING and not MISNAMED:
    print("  every source in SOURCES exists under the name given.")
else:
    print(f"\n  {len(MISNAMED)} likely misnamed, {len(MISSING)} genuinely absent.")
    print("  A genuinely absent table that ps1_features.py reads means that feature family")
    print("  has been silently missing from every run — the reads are inside try/except.")


In [ ]:
# ── CELL 2 ─ source currency: UC table vs the S3 export, scored per source ────
# (layer, table, date column, tier)
#   spine    = the label source and the training spine. These set the HARD ceiling:
#              past them there is nothing to train on at all.
#   core     = read on every run. Stale here means stale features on every later row.
#   optional = flag-gated. Stale here matters only if you turn that flag on.
#   ref      = dimension/snapshot, no date column. Judge by row count.
SOURCES = [
    ("silver", "device_event_enriched",          "transit_day",       "spine"),
    ("gold",   "device_ps1_daily",               "transit_day",       "spine"),
    ("silver", "device_outage",                  "transit_day",       "core"),
    ("silver", "device_mttr",                    "failure_date",       "core"),
    ("silver", "usage_lifecycle_daily",          "transit_day",       "core"),
    ("silver", "read_tap_daily",                 "transit_day",       "core"),
    ("silver", "read_tap_device_daily",          "transit_day",       "core"),
    ("silver", "device_incident_features_daily", "transit_day",       "core"),
    ("silver", "tap_event_daily",                "transit_day",       "core"),
    ("silver", "use_revenue_daily",              "transit_day",       "core"),
    ("silver", "incident_root_cause",            "transit_day",       "optional"),
    ("silver", "incident_history",               "incident_date",       "optional"),
    ("silver", "metric_daily",                   "transit_day",       "optional"),
    ("silver", "kpi_daily",                      "transit_day",       "optional"),
    ("silver", "kpi_avail_enriched",             "transit_day",       "optional"),
    ("silver", "warnings_daily",                 "event_date",       "optional"),
    ("silver", "station_network_daily",          "transit_day",       "optional"),
    ("silver", "device_uptime_intervals",        "eod_date",          "optional"),
    ("silver", "maintenance_ledger",             "ledger_date",       "optional"),
    ("silver", "hw_config_current",              "LAST_REPORTED_DTM", "optional"),
    ("gold",   "device_ps2_chains",              "transit_day",       "optional"),
    ("gold",   "device_ps4_hourly",              "transit_day",       "optional"),
    ("silver", "dim_device",                     None,                "ref"),
    ("silver", "dim_event_matrix",               None,                "ref"),
    ("silver", "dim_event_type",                 None,                "ref"),
]

print("=" * 108)
print(f"1. SOURCE CURRENCY vs {EXPECTED_THROUGH}    (UC = the live table, S3 = what PS1 actually reads)")
print("=" * 108)
print(f"  {'tier':9s} {'source':36s} {'UC max':>12s} {'UC rows':>14s} {'S3 max':>12s} {'S3 rows':>14s}  verdict")

def _measure(df, datecol):
    """Row count, then the max of the date column resolved case-insensitively.
    A missing date column is reported by name, never raised as if the table were absent."""
    rows = df.count()
    if not datecol:
        return rows, None, ""
    real = {c.casefold(): c for c in df.columns}.get(datecol.casefold())
    if real is None:
        return rows, None, f"NO DATE COL {datecol!r}"
    return rows, df.agg(F.max(F.to_date(F.col(real)))).collect()[0][0], ""


CURRENCY = []
for layer, tbl, datecol, tier in SOURCES:
    uc_max = s3_max = None
    uc_rows = s3_rows = None
    # Count and date are measured SEPARATELY. On the first run the date aggregation
    # raised for three tables whose date column is not transit_day, and the except
    # overwrote a good row count with ERR -- so tables that were fine read as missing.
    note = ""
    try:
        uc_rows, uc_max, _n = _measure(spark.table(f"{CATALOG}.{layer}.{tbl}"), datecol)
        note = note or _n
    except Exception as exc:
        uc_rows = f"ERR {type(exc).__name__}"
    try:
        root = GOLD_S3 if layer == "gold" else SILVER_S3
        s3_rows, s3_max, _n = _measure(spark.read.parquet(f"{root}/{tbl}/"), datecol)
        note = note or _n
    except Exception as exc:
        s3_rows = f"ERR {type(exc).__name__}"

    verdict = ""
    if uc_max and s3_max and uc_max != s3_max:
        verdict = f"EXPORT {(uc_max - s3_max).days}d BEHIND UC"
    elif datecol and s3_max:
        n = (EXPECTED_THROUGH - s3_max).days
        verdict = "current" if n <= 0 else f"{n}d BEHIND"
    elif not datecol:
        verdict = "reference (no date)"
    elif note:
        verdict = note
    elif isinstance(uc_rows, str) or isinstance(s3_rows, str):
        verdict = "UNREADABLE"

    CURRENCY.append((f"{layer}.{tbl}", uc_max, uc_rows, s3_max, s3_rows, verdict, tier))
    print(f"  {tier:9s} {layer + '.' + tbl:36s} {str(uc_max):>12s} {str(uc_rows):>14s} "
          f"{str(s3_max):>12s} {str(s3_rows):>14s}  {verdict}")

# ---- ceilings, by tier -------------------------------------------------------
# One stale OPTIONAL source must not drag the verdict back five months. It only
# constrains a run that turns that flag on.
def _ceiling(tiers):
    ds = [d for _, _, _, d, _, _, ti in CURRENCY if d and ti in tiers]
    return min(ds) if ds else None

SPINE_CEIL = _ceiling({"spine"})
CORE_CEIL  = _ceiling({"spine", "core"})
ALL_CEIL   = _ceiling({"spine", "core", "optional"})
WINDOW_CEILING = CORE_CEIL          # the one to actually plan against

print()
print("=" * 108)
print("  CEILINGS")
print("=" * 108)
print(f"  label + spine only              : {SPINE_CEIL}   <- past this there is nothing to train on")
print(f"  + every always-read feature     : {CORE_CEIL}   <- PLAN AGAINST THIS")
print(f"  + every flag-gated feature too  : {ALL_CEIL}")
if SPINE_CEIL and CORE_CEIL and SPINE_CEIL != CORE_CEIL:
    print(f"\n  Ending at {SPINE_CEIL} instead of {CORE_CEIL} costs "
          f"{(SPINE_CEIL - CORE_CEIL).days} days of stale features on every row in between.")
for _tier in ("spine", "core", "optional"):
    lag = sorted([(t, (EXPECTED_THROUGH - dd).days, dd)
                  for t, _, _, dd, _, _, ti in CURRENCY
                  if dd and ti == _tier and (EXPECTED_THROUGH - dd).days > 0],
                 key=lambda x: -x[1])
    if lag:
        print(f"\n  {_tier.upper()} sources behind {EXPECTED_THROUGH}:")
        for t, n, dd in lag:
            print(f"    {t:38s} {str(dd):>12s}  {n:>5d}d")


In [ ]:
# ── CELL 3 ─ what the label source itself covers, per fleet ──────────────────
print("=" * 108)
print("2. THE LABEL SOURCE  (silver.device_event_enriched, hardware-OOS Set events)")
print("=" * 108)

dee = spark.table(f"{CATALOG}.silver.device_event_enriched")
_cols = {c.casefold(): c for c in dee.columns}
_oos  = _cols.get("is_hardware_oos_event")
_st   = _cols.get("event_state_type_name")
_cat  = _cols.get("mars_device_category")
_dtm  = _cols.get("event_dtm")
print(f"  columns resolved: oos={_oos!r} state={_st!r} cat={_cat!r} dtm={_dtm!r}")

fail_days = (dee
    .where(F.col(_oos) == True)
    .where(F.upper(F.col(_st)) == "SET")
    .select(F.col("DEVICE_ID").alias("DEVICE_ID"),
            F.col(_cat).alias("cat"),
            F.to_date(F.col(_dtm)).alias("failure_date"))
    .distinct()).cache()

cov = (fail_days.groupBy("cat")
       .agg(F.min("failure_date").alias("first_day"),
            F.max("failure_date").alias("last_day"),
            F.countDistinct("DEVICE_ID").alias("devices"),
            F.count(F.lit(1)).alias("failure_days"))
       .collect())
LABEL_COVERAGE = {}
print(f"\n  {'fleet':12s} {'first':>12s} {'last':>12s} {'devices':>9s} {'failure-days':>14s}")
for r in sorted(cov, key=lambda x: str(x["cat"])):
    LABEL_COVERAGE[r["cat"]] = r["last_day"]
    print(f"  {str(r['cat']):12s} {str(r['first_day']):>12s} {str(r['last_day']):>12s} "
          f"{r['devices']:>9,} {r['failure_days']:>14,}")

print("\n  Episode starts, sessionised on a %d-day gap (the PS1 default rule):" % SESSION_GAP_DAYS)
_w = Window.partitionBy("DEVICE_ID").orderBy("failure_date")
episodes = (fail_days
    .withColumn("_prev", F.lag("failure_date").over(_w))
    .where(F.col("_prev").isNull()
           | (F.datediff(F.col("failure_date"), F.col("_prev")) > SESSION_GAP_DAYS))
    .drop("_prev")).cache()
EPISODES = {r["cat"]: r["n"] for r in
            episodes.groupBy("cat").agg(F.count(F.lit(1)).alias("n")).collect()}
for k in sorted(EPISODES, key=str):
    print(f"    {str(k):12s} {EPISODES[k]:>10,} episode starts")


In [ ]:
# ── CELL 4 ─ horizon arithmetic: base rate and required AUC, no training ─────
# Required AUC for the contract pair, under a binormal assumption:
#   FPR_max = R * p * (1 - P) / (P * (1 - p))      P = precision, R = recall, p = base rate
#   d'      = z(R) - z(FPR_max);   AUC = Phi(d' / sqrt(2))
import math
_N = NormalDist()

def required_auc(p, P=REQ_PRECISION, R=REQ_RECALL):
    if not (0 < p < 1):
        return None
    fpr = R * p * (1 - P) / (P * (1 - p))
    if fpr <= 0:
        return None
    if fpr >= 1:
        return 0.5                      # no constraint: degenerate
    d = _N.inv_cdf(R) - _N.inv_cdf(fpr)
    return _N.cdf(d / math.sqrt(2))

print("=" * 108)
print("3. HORIZON ARITHMETIC  (positives are device-days inside the lookahead of an episode start)")
print("=" * 108)

spine = spark.table(f"{CATALOG}.gold.device_ps1_daily")
_scat = {c.casefold(): c for c in spine.columns}.get("mars_device_category")

HORIZON_TABLE = []
for cat in FLEETS:
    last_day = LABEL_COVERAGE.get(cat)
    if last_day is None:
        print(f"\n  {cat}: no failure days found, skipped")
        continue
    ep = episodes.where(F.col("cat") == cat).select("DEVICE_ID", "failure_date")
    sp = spine.where(F.col(_scat) == cat).select("DEVICE_ID", "transit_day")
    denom = sp.count()
    print(f"\n  {cat}   device-days in gold spine: {denom:,}   episodes: {EPISODES.get(cat, 0):,}")
    print(f"  {'horizon':>9s} {'positives':>12s} {'base rate':>11s} {'AUC needed':>11s}  note")
    for H in HORIZONS:
        seq = spark.range(1, H + 1).select(F.col("id").cast("int").alias("n"))
        pos = (ep.crossJoin(seq)
                 .withColumn("label_day", F.date_sub(F.col("failure_date"), F.col("n")))
                 .select("DEVICE_ID", "label_day").distinct())
        n_pos = (sp.join(pos,
                         (sp.DEVICE_ID == pos.DEVICE_ID) & (sp.transit_day == pos.label_day),
                         "inner").count())
        p = n_pos / denom if denom else 0.0
        auc = required_auc(p)
        note = ""
        if p >= 0.5:
            note = "DEGENERATE - over half of all device-days are positive"
        elif auc is not None and auc <= 0.5:
            note = "DEGENERATE - the precision constraint is unbinding"
        HORIZON_TABLE.append((cat, H, n_pos, denom, p, auc, note))
        print(f"  {H:>7d}d {n_pos:>12,} {p:>10.4%} "
              f"{('%.4f' % auc) if auc else '     n/a':>11s}  {note}")

print("\n  NOTE: auc_needed above is the PRECISION>=0.90 + RECALL>=0.80 pair ONLY. Two further")
print("  MUST-MEET criteria cap the usable base rate independently -- FPR<=0.10 caps p near 0.529,")
print("  accuracy>=0.90 caps it lower -- so a LOW auc_needed at a HIGH base rate does not mean the")
print("  contract is met there. It means a different criterion is what fails.")
print("\n  Lowest bar the contract can present once ALL of them apply: 0.9631")
print("  With the Table 14 accuracy relaxation live:                0.9353")
print("  Measured on the honest label, 7-day: GATE 0.769  TVM 0.832  VALIDATOR 0.910")


In [ ]:
# ── CELL 5 ─ paste this block back ───────────────────────────────────────────
print("8<" + "-" * 104)
print(f"PS1 DATA CURRENCY + HORIZON CHECK   expected_through={EXPECTED_THROUGH}")
print("-" * 106)
print("[CURRENCY]  tier | source | uc_max | uc_rows | s3_max | s3_rows | verdict")
for t, um, ur, sm, sr, v, ti in CURRENCY:
    print(f"  {ti} | {t} | {um} | {ur} | {sm} | {sr} | {v}")
print(f"[CEILINGS] spine={SPINE_CEIL} core={CORE_CEIL} all={ALL_CEIL}")
print(f"[PREFLIGHT] misnamed={MISNAMED} missing={MISSING}")
print("[LABEL_COVERAGE]  fleet | last_failure_day | episode_starts")
for k in sorted(LABEL_COVERAGE, key=str):
    print(f"  {k} | {LABEL_COVERAGE[k]} | {EPISODES.get(k, 0)}")
print("[HORIZON]  fleet | horizon_days | positives | device_days | base_rate | auc_needed | note")
for cat, H, n_pos, denom, p, auc, note in HORIZON_TABLE:
    print(f"  {cat} | {H} | {n_pos} | {denom} | {p:.6f} | "
          f"{('%.4f' % auc) if auc else 'n/a'} | {note}")
print("-" * 106 + ">8")
